# Kaggle Auto-Train Notebook

Notebook nay duoc viet lai theo thong tin Kaggle moi trong `INFO.md`.

Flow mac dinh:
- tim source code repo trong `/kaggle/input`
- copy repo vao `/kaggle/working`
- dung moi truong Python mac dinh cua Kaggle, khong tao lai venv va khong cai lai thu vien
- tu dong chon file train / validation / test hop le trong ViANLI
- train `mBERT + FFN` voi `configs/default.yaml` + `configs/kaggle.yaml`


## 1. Cau hinh notebook

Cell nay tap trung toan bo bien de sua nhanh: mount hint cho source code, mount hint cho dataset, ten run, co bat pytest hay khong, co zip output hay khong, va cac override bo sung neu can.


In [ ]:
from pathlib import Path

REPO_SOURCE_HINT = "/kaggle/input/datasets/khoa05ai/fu-s7dat-tuningmodel/vnnli_engram_moe-main"
DATASET_DIR_HINT = "/kaggle/input/datasets/khoa05ai/fu-s7dat-vieanli/vianli_kaggle"

WORK_ROOT = Path("/kaggle/working")
REPO_DIR = WORK_ROOT / "vnnli_engram_moe"
OUTPUT_ROOT = WORK_ROOT / "outputs" / "runs"

BASE_CONFIG = "configs/default.yaml"
USER_CONFIG = "configs/kaggle.yaml"
MODEL_CONFIG = "configs/models/mbert_cased.yaml"
TRAIN_SCRIPT = REPO_DIR / "scripts/train.py"
BASE_CONFIG_PATH = REPO_DIR / BASE_CONFIG
USER_CONFIG_PATH = REPO_DIR / USER_CONFIG
MODEL_CONFIG_PATH = REPO_DIR / MODEL_CONFIG
RUN_NAME = "kaggle_mbert_vianli"

RUN_PYTEST = False
MAKE_OUTPUT_ZIP = False
USE_FP16 = True
FORCE_RECOPY_REPO = True
EXTRA_OVERRIDES = []


## 2. Tim mounted repo va mounted dataset

Cell nay tim repo va du lieu ben trong `/kaggle/input`, in ra cac path duoc resolve, sau do copy source code sang `/kaggle/working` de training co the ghi output mot cach binh thuong.


In [ ]:
import shlex
import shutil
import subprocess
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def shell_join(parts):
    return " ".join(shlex.quote(str(part)) for part in parts)

def run(parts, *, cwd=None, capture=False):
    command = parts if isinstance(parts, str) else shell_join(parts)
    print(f"$ {command}")
    try:
        result = subprocess.run(
            command,
            cwd=cwd,
            shell=True,
            check=True,
            text=True,
            capture_output=capture,
        )
    except subprocess.CalledProcessError as error:
        if error.stdout:
            print(error.stdout)
        if error.stderr:
            print(error.stderr)
        raise
    if capture:
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
    return result

def preview_dir(path, limit=20):
    path = Path(path)
    if not path.exists():
        print(f"[missing] {path}")
        return
    print(f"Contents of {path}:")
    entries = sorted(path.iterdir(), key=lambda item: item.name)
    for item in entries[:limit]:
        suffix = "/" if item.is_dir() else ""
        print(f"- {item.name}{suffix}")
    if len(entries) > limit:
        print(f"... and {len(entries) - limit} more")

def iter_repo_candidates():
    seen = set()
    if REPO_SOURCE_HINT:
        candidate = Path(REPO_SOURCE_HINT)
        if candidate not in seen:
            seen.add(candidate)
            yield candidate
    for candidate in INPUT_ROOT.rglob("vnnli_engram_moe-main"):
        if candidate.is_dir() and candidate not in seen:
            seen.add(candidate)
            yield candidate
    for pyproject in INPUT_ROOT.rglob("pyproject.toml"):
        candidate = pyproject.parent
        if candidate in seen:
            continue
        if (candidate / "scripts/train.py").exists() and (candidate / "configs/default.yaml").exists():
            seen.add(candidate)
            yield candidate

def resolve_repo_source_dir():
    for candidate in iter_repo_candidates():
        if candidate.is_dir() and (candidate / "pyproject.toml").exists() and (candidate / "scripts/train.py").exists():
            return candidate
    preview_dir(INPUT_ROOT, limit=50)
    raise FileNotFoundError(
        "Could not locate the source repo under /kaggle/input. Update REPO_SOURCE_HINT in the config cell."
    )

def iter_data_candidates():
    seen = set()
    if DATASET_DIR_HINT:
        candidate = Path(DATASET_DIR_HINT)
        if candidate not in seen:
            seen.add(candidate)
            yield candidate
    for candidate in INPUT_ROOT.rglob("vianli_kaggle"):
        if candidate.is_dir() and candidate not in seen:
            seen.add(candidate)
            yield candidate
    for candidate in INPUT_ROOT.rglob("*"):
        if not candidate.is_dir() or candidate in seen:
            continue
        has_train = any((candidate / name).exists() for name in ["train.jsonl", "train.csv"])
        has_validation = any((candidate / name).exists() for name in ["validation.jsonl", "validation.csv", "dev.jsonl", "dev.csv"])
        has_test = any((candidate / name).exists() for name in ["test.jsonl", "test.csv"])
        if has_train and has_validation and has_test:
            seen.add(candidate)
            yield candidate

def resolve_data_dir():
    for candidate in iter_data_candidates():
        if candidate.is_dir():
            return candidate
    preview_dir(INPUT_ROOT, limit=50)
    raise FileNotFoundError(
        "Could not locate the ViANLI dataset directory under /kaggle/input. Update DATASET_DIR_HINT in the config cell."
    )

def resolve_split_file(data_dir, split_name):
    candidates = {
        "train": ["train.jsonl", "train.csv"],
        "validation": ["validation.jsonl", "validation.csv", "dev.jsonl", "dev.csv"],
        "test": ["test.jsonl", "test.csv"],
    }[split_name]
    for name in candidates:
        path = data_dir / name
        if path.exists():
            return path
    unsupported_json = data_dir / f"{split_name}.json"
    if unsupported_json.exists():
        raise ValueError(
            f"Found {unsupported_json}, but the repo only supports JSONL/CSV/Parquet for direct file loading. "
            "Please keep the CSV or JSONL split alongside it."
        )
    raise FileNotFoundError(f"Could not find a supported file for split '{split_name}' in {data_dir}")

repo_source_dir = resolve_repo_source_dir()
data_dir = resolve_data_dir()
train_file = resolve_split_file(data_dir, "train")
validation_file = resolve_split_file(data_dir, "validation")
test_file = resolve_split_file(data_dir, "test")

print(f"Resolved repo source: {repo_source_dir}")
print(f"Resolved data dir: {data_dir}")
print(f"Train file: {train_file}")
print(f"Validation file: {validation_file}")
print(f"Test file: {test_file}")
preview_dir(data_dir)

if REPO_DIR.exists() and FORCE_RECOPY_REPO:
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    shutil.copytree(repo_source_dir, REPO_DIR)

print(f"Working repo: {REPO_DIR}")
preview_dir(REPO_DIR)


## 3. Kiem tra moi truong mac dinh cua Kaggle

Cell nay khong tao `.venv` va khong cai them thu vien. No dung truc tiep `sys.executable` cua notebook hien tai de check `train.py --help` bang absolute path, in version Python / Torch / Transformers, kiem tra GPU, va cho phep chay `pytest` neu ban bat `RUN_PYTEST = True`.


In [ ]:
import subprocess
import sys

python_bin = sys.executable
print(f"Using Kaggle default python: {python_bin}")

run([python_bin, TRAIN_SCRIPT, "--help"], cwd=REPO_DIR)

version_script = """
import platform
import sys
import torch
import transformers
print('python', sys.version.split()[0])
print('platform', platform.platform())
print('torch', torch.__version__)
print('transformers', transformers.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda_device', torch.cuda.get_device_name(0))
"""
run([python_bin, "-c", version_script], cwd=REPO_DIR)
subprocess.run(["nvidia-smi"], check=False)

if RUN_PYTEST:
    run([python_bin, "-m", "pytest"], cwd=REPO_DIR)
else:
    print("Skipping pytest to keep the notebook focused on training. Set RUN_PYTEST = True to enable it.")


## 4. Chay train mac dinh voi mBERT + FFN

Cell nay dung Python mac dinh cua Kaggle de goi `scripts/train.py` bang absolute path, truyen `default.yaml`, `kaggle.yaml`, `mbert_cased.yaml`, va 3 split cua ViANLI. Neu can tat fp16 hoac them override khac, sua ngay trong cell cau hinh dau tien.


In [ ]:
import sys
from pathlib import Path

python_bin = sys.executable
train_command = [
    python_bin,
    str(TRAIN_SCRIPT),
    "--config",
    str(BASE_CONFIG_PATH),
    "--user-config",
    str(USER_CONFIG_PATH),
    "--model-config",
    str(MODEL_CONFIG_PATH),
    "--train-file",
    str(train_file),
    "--validation-file",
    str(validation_file),
    "--test-file",
    str(test_file),
    "--output-dir",
    str(OUTPUT_ROOT),
    "--run-name",
    RUN_NAME,
]

overrides = list(EXTRA_OVERRIDES)
if not USE_FP16:
    overrides.append("training.fp16=false")

for override in overrides:
    train_command.extend(["--override", override])

train_result = run(train_command, cwd=REPO_DIR, capture=True)
stdout_lines = [line.strip() for line in train_result.stdout.splitlines() if line.strip()]
RUN_DIR = Path(stdout_lines[-1]) if stdout_lines else None
if RUN_DIR is None or not RUN_DIR.exists():
    run_dirs = sorted(OUTPUT_ROOT.glob("*"), key=lambda path: path.stat().st_mtime)
    if not run_dirs:
        raise FileNotFoundError(f"No run directory was created under {OUTPUT_ROOT}")
    RUN_DIR = run_dirs[-1]

print(f"RUN_DIR = {RUN_DIR}")


## 5. Tong ket artifact sau khi train

Cell cuoi cung in ra thu muc run, preview file ben trong, hien `train_metrics.json`, `dev_metrics.json`, `test_metrics.json`, `run_metadata.json`, va co tuy chon zip toan bo output neu ban bat `MAKE_OUTPUT_ZIP = True`.


In [ ]:
import json
import shutil

print(f"Final run directory: {RUN_DIR}")
preview_dir(RUN_DIR, limit=50)

for name in ["train_metrics.json", "dev_metrics.json", "test_metrics.json", "run_metadata.json"]:
    path = RUN_DIR / name
    if path.exists():
        print(f"\n== {name} ==")
        print(json.dumps(json.loads(path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
    else:
        print(f"Missing expected artifact: {path}")

final_model_dir = RUN_DIR / "final_model"
print(f"\nFinal model dir: {final_model_dir}")
preview_dir(final_model_dir, limit=50)

if MAKE_OUTPUT_ZIP:
    archive_base = WORK_ROOT / "vnnli_outputs"
    archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=WORK_ROOT, base_dir="outputs")
    print(f"Created zip archive: {archive_path}")
else:
    print("MAKE_OUTPUT_ZIP is False, skipping zip export.")
